# Sketch → Image with Generative AI
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malikishaq881/sketch2image/blob/main/Sketch2Image_Colab.ipynb)

**Author:** Malik Muhammad Ishaq  
**Program:** International Master of Biometrics and Intelligent Vision, UPEC  
**Project:** Generative AI – *Bringing Sketches to Life*  
**Method:** Stable Diffusion 1.5 + ControlNet Scribble  
**Repository:** https://github.com/malikishaq881/sketch2image

Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

## 1. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found: select Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the code and install dependencies

In [ ]:
%cd /content
!rm -rf sketch2image
!git clone -q https://github.com/malikishaq881/sketch2image.git
%cd /content/sketch2image
!pip install -q -r requirements.txt
!ls

## 3. Input sketches
By default the four sketches from the repository are used. Set `USE_OWN_SKETCHES = True` to upload your own images (dark lines on a light background) and describe each one.

In [ ]:
import json
import os
import shutil
from pathlib import Path

from PIL import Image

USE_OWN_SKETCHES = False

# Used only when USE_OWN_SKETCHES = True: file name (without extension) -> subject description
MY_PROMPTS = {
    "sketch1": "a futuristic building",
    "sketch2": "a fantasy character",
    "sketch3": "a modern product design",
}

if USE_OWN_SKETCHES:
    from google.colab import files

    shutil.rmtree("sketches", ignore_errors=True)
    os.makedirs("sketches")
    for name in files.upload():
        shutil.move(name, f"sketches/{name}")
    Path("prompts.json").write_text(json.dumps(MY_PROMPTS, indent=2))

prompts = json.loads(Path("prompts.json").read_text())
sketch_files = sorted(Path("sketches").glob("*.*"))
print("Sketches:", [p.name for p in sketch_files])
print(json.dumps(prompts, indent=2))

strip = Image.new("RGB", (256 * len(sketch_files), 256), "white")
for i, p in enumerate(sketch_files):
    strip.paste(Image.open(p).convert("RGB").resize((256, 256)), (256 * i, 0))
strip

## 4. Run the experiments
The full run takes about 20–30 minutes on a T4 (the first run also downloads about 5 GB of models). Set `QUICK = True` for a short test first.

In [ ]:
QUICK = False

flag = "--quick" if QUICK else ""
!python run_experiments.py --sketches sketches --prompts prompts.json --out results {flag}

## 5. Results: sketches and generated images

In [ ]:
import glob

from IPython.display import Image as show, display

print("Overview: sketch | photorealistic | cyberpunk | watercolor")
display(show("results/overview.png"))

for path in sorted(glob.glob("results/*/grid_*.png")):
    print(path)
    display(show(path))

## 6. Metrics

In [ ]:
import pandas as pd

display(show("results/plot_cond_scale.png"))
pd.read_csv("results/summary.csv", index_col=[0, 1])

## 7. Download the results
Upload the unzipped `results` folder to your GitHub repository (Add file → Upload files).

In [ ]:
!zip -qr results.zip results
from google.colab import files

files.download("results.zip")

## 8. Optional: web demo
Opens a Gradio interface. Click the public link printed below, upload a sketch, describe it and press *Generate*.

In [ ]:
!python app.py --share